# Pré-processamento e Feature Engineering — House Prices

**Fase 2** do Projeto 2 — UNIFOR. Partindo das decisões da EDA (`01_eda.ipynb`), este notebook:
1. Cria features derivadas que capturam informação latente nos dados;
2. Valida o ganho dessas features via correlação;
3. Constrói o `ColumnTransformer` de pré-processamento;
4. Explica a estratégia de transformação do alvo;
5. Discute prevenção de *data leakage*.

O código aqui é a **fonte de verdade** — a função `engenharia_features` será copiada verbatim para `submissao/pipeline.py` na Fase 3.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

df_raw = pd.read_csv('treino.csv')
print(f'Treino bruto: {df_raw.shape[0]} linhas x {df_raw.shape[1]} colunas')

Treino bruto: 1168 linhas x 81 colunas


---
## 1. Feature Engineering

### Motivação

A EDA mostrou que `OverallQual` e `GrLivArea` lideram as correlações com `SalePrice`. Isso revela um padrão: **imóveis maiores e de maior qualidade valem mais**. As features brutas, porém, fragmentam a informação: área do porão, primeiro andar e segundo andar estão em colunas separadas, mas o que importa para o comprador é a **área total**. Features compostas capturem esse sinal de forma mais direta.

### Descrição de cada feature criada

| Feature | Fórmula | Por que existe |
|---------|---------|----------------|
| `TotalSF` | `TotalBsmtSF + 1stFlrSF + 2ndFlrSF` | Área total do imóvel em todos os andares — sinal mais forte que cada parcela isolada. |
| `TotalBath` | `FullBath + 0.5·HalfBath + BsmtFullBath + 0.5·BsmtHalfBath` | Banheiros completos valem mais que lavabos; agrega o total ponderado. |
| `HouseAge` | `YrSold − YearBuilt` (≥0) | Imóveis mais antigos tendem a valer menos; captura depreciação por idade. |
| `RemodAge` | `YrSold − YearRemodAdd` (≥0) | Uma reforma recente recupera parte do valor perdido pela idade. |
| `TotalPorchSF` | soma de todas as áreas de varanda/deck | Área de lazer externa agrega valor, mas cada tipo isolado tem correlação fraca. |
| `HasPool` | `PoolArea > 0` → 1/0 | A presença de piscina é um *premium* independente do tamanho; flag binária é suficiente. |
| `HasGarage` | `GarageArea > 0` → 1/0 | Dicotomiza a existência de garagem, separando do seu tamanho (`GarageArea`). |
| `Has2ndFloor` | `2ndFlrSF > 0` → 1/0 | Casas de dois andares têm perfil de preço diferente das térreas, além da área. |
| `HasBsmt` | `TotalBsmtSF > 0` → 1/0 | Presença de porão é informação de estrutura; complementa `TotalBsmtSF`. |
| `HasFireplace` | `Fireplaces > 0` → 1/0 | Lareira é um *amenity* de status; a simples presença já eleva o preço. |

### Nota de design

A função usa um auxiliar `col()` que retorna zeros se a coluna não existir. Isso torna o código tolerante à base de teste pública/secreta, que pode não ter a coluna `SalePrice` ou outras colunas opcionais — sem quebrar o pipeline.

In [2]:
def engenharia_features(df):
    df = df.copy()
    def col(nome):
        if nome in df.columns:
            return pd.to_numeric(df[nome], errors='coerce').fillna(0)
        return pd.Series(0, index=df.index)
    df['TotalSF']      = col('TotalBsmtSF') + col('1stFlrSF') + col('2ndFlrSF')
    df['TotalBath']    = col('FullBath') + 0.5*col('HalfBath') + col('BsmtFullBath') + 0.5*col('BsmtHalfBath')
    df['HouseAge']     = (col('YrSold') - col('YearBuilt')).clip(lower=0)
    df['RemodAge']     = (col('YrSold') - col('YearRemodAdd')).clip(lower=0)
    df['TotalPorchSF'] = col('OpenPorchSF')+col('EnclosedPorch')+col('3SsnPorch')+col('ScreenPorch')+col('WoodDeckSF')
    df['HasPool']      = (col('PoolArea')>0).astype(int)
    df['HasGarage']    = (col('GarageArea')>0).astype(int)
    df['Has2ndFloor']  = (col('2ndFlrSF')>0).astype(int)
    df['HasBsmt']      = (col('TotalBsmtSF')>0).astype(int)
    df['HasFireplace'] = (col('Fireplaces')>0).astype(int)
    if 'Id' in df.columns:
        df = df.drop(columns=['Id'])
    return df

In [3]:
df = engenharia_features(df_raw)

novas = ['TotalSF','TotalBath','HouseAge','RemodAge','TotalPorchSF',
         'HasPool','HasGarage','Has2ndFloor','HasBsmt','HasFireplace']

print(f'Shape após feature engineering: {df.shape}')
print(f'Novas colunas criadas: {len(novas)}')
df[novas].head(10)

Shape após feature engineering: (1168, 90)
Novas colunas criadas: 10


,TotalSF,TotalBath,HouseAge,RemodAge,TotalPorchSF,HasPool,HasGarage,Has2ndFloor,HasBsmt,HasFireplace
0,2628,2.0,53,53,250,0,1,0,1,0
1,2370,2.5,16,15,40,0,1,1,1,1
2,1592,1.0,98,58,492,0,0,0,1,0
3,2499,2.5,70,57,264,0,1,1,1,1
4,2717,2.0,86,60,242,0,1,1,1,1
5,1788,2.0,34,34,256,0,1,0,1,0
6,2244,2.5,4,4,138,0,1,1,1,1
7,1950,2.0,88,3,96,0,1,1,1,0
8,2844,2.0,27,27,276,0,1,0,1,1
9,2992,2.0,0,0,298,0,1,0,1,0


---
## 2. Validação das Novas Features

Para confirmar que as features criadas agregam valor preditivo, calculamos a correlação de Pearson de cada uma com `SalePrice`. Esperamos que `TotalSF` e `TotalBath` — que agregam múltiplas colunas brutas — apareçam entre os valores mais altos, superando as parcelas individuais.

In [4]:
corr_novas = (
    df[novas + ['SalePrice']]
    .corr()['SalePrice']
    .drop('SalePrice')
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)
corr_novas.columns = ['Feature', '|Correlação com SalePrice|']
corr_novas['|Correlação com SalePrice|'] = corr_novas['|Correlação com SalePrice|'].round(4)

# Referência: correlação de GrLivArea e TotalBsmtSF originais
ref = df[['GrLivArea','TotalBsmtSF','1stFlrSF','2ndFlrSF','SalePrice']].corr()['SalePrice'].drop('SalePrice').abs()

print('=== Correlação das 10 novas features com SalePrice ===')
print(corr_novas.to_string(index=False))
print()
print('=== Referência: features brutas equivalentes ===')
print(f'  GrLivArea   : {ref["GrLivArea"]:.4f}')
print(f'  TotalBsmtSF : {ref["TotalBsmtSF"]:.4f}')
print(f'  1stFlrSF    : {ref["1stFlrSF"]:.4f}')
print(f'  2ndFlrSF    : {ref["2ndFlrSF"]:.4f}')

=== Correlação das 10 novas features com SalePrice ===
     Feature  |Correlação com SalePrice|
     TotalSF                      0.7656
   TotalBath                      0.6213
    HouseAge                      0.5164
    RemodAge                      0.5097
HasFireplace                      0.4706
TotalPorchSF                      0.3885
   HasGarage                      0.2477
     HasBsmt                      0.1556
 Has2ndFloor                      0.1297
     HasPool                      0.1174

=== Referência: features brutas equivalentes ===
  GrLivArea   : 0.6957
  TotalBsmtSF : 0.5978
  1stFlrSF    : 0.5879
  2ndFlrSF    : 0.3140


**Interpretação:** `TotalSF` deve superar `TotalBsmtSF` e `1stFlrSF` isolados — a combinação captura mais sinal. `TotalBath` consolida quatro colunas de banheiro num único preditor mais limpo. As flags binárias (`HasFireplace`, `HasGarage` etc.) têm correlação menor individualmente, mas adicionam informação ortogonal que beneficia modelos de árvore (que criam cortes binários justamente sobre esses valores).

---
## 3. Pré-processamento com `ColumnTransformer`

### Estratégia

Colunas numéricas e categóricas precisam de tratamentos diferentes:

| Etapa | Numéricas | Categóricas | Justificativa |
|-------|-----------|-------------|---------------|
| Imputação | `SimpleImputer(strategy='median')` | `SimpleImputer(strategy='constant', fill_value='Ausente')` | Mediana é robusta a outliers; `'Ausente'` preserva o significado semântico dos NaN identificados na EDA (casa sem piscina, sem garagem etc.). |
| Encoding | `StandardScaler()` | `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` | Padronização ajuda modelos lineares e é inofensiva a árvores; `handle_unknown='ignore'` evita quebra se a base secreta contiver categoria não vista no treino. |

Usamos `make_column_selector` para identificar automaticamente as colunas por tipo — assim, novas features numéricas criadas por `engenharia_features` são incluídas sem alterar o código do transformer.

In [5]:
# Separamos X e y antes de ajustar o transformer
X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Ausente')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, make_column_selector(dtype_include=np.number)),
        ('cat', cat_pipeline, make_column_selector(dtype_include=object)),
    ],
    remainder='drop',
)

X_proc = preprocessor.fit_transform(X)

n_num = len(make_column_selector(dtype_include=np.number)(X))
n_cat = preprocessor.named_transformers_['cat']['ohe'].get_feature_names_out().shape[0]

print(f'Colunas numéricas antes do OHE : {n_num}')
print(f'Colunas geradas pelo OHE       : {n_cat}')
print(f'Dimensão final de X_proc       : {X_proc.shape}')
print(f'  → {X_proc.shape[0]} amostras x {X_proc.shape[1]} features')

Colunas numéricas antes do OHE : 46
Colunas geradas pelo OHE       : 265
Dimensão final de X_proc       : (1168, 311)
  → 1168 amostras x 311 features


O `OneHotEncoder` expande cada coluna categórica em tantas colunas binárias quantas são suas categorias únicas (menos 1 se `drop='first'` — aqui usamos `drop=None` por compatibilidade com modelos de árvore que não sofrem de multicolinearidade). O resultado é uma matriz densa e totalmente numérica, pronta para qualquer estimador do scikit-learn.

---
## 4. Alvo em `log1p` — `TransformedTargetRegressor`

### Por que embutir a transformação no pipeline

A EDA mostrou que `log1p(SalePrice)` tem skewness ≈ 0.12 contra 1.74 do valor bruto. Além disso, a métrica da competição é **RMSLE**, que é equivalente a **RMSE calculado na escala log**. Portanto:

$$\text{RMSLE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (\log(1+\hat{y}_i) - \log(1+y_i))^2}$$

Treinar diretamente minimizando RMSE em `log1p(y)` é matematicamente equivalente a minimizar RMSLE.

### Implementação na Fase 3

Na modelagem usaremos `TransformedTargetRegressor` do scikit-learn:

```python
from sklearn.compose import TransformedTargetRegressor

modelo_final = TransformedTargetRegressor(
    regressor   = pipeline_com_preprocessor,
    func        = np.log1p,
    inverse_func= np.expm1,
)
```

**Vantagem crítica:** o `TransformedTargetRegressor` aplica `log1p` automaticamente antes de treinar e `expm1` automaticamente ao predizer. Isso garante que o pipeline de produção nunca "esqueça" o `expm1` — o erro clássico de entregar predições em escala log em vez de dólares.

In [6]:
# Demonstração: equivalência entre treinar em log1p e usar TransformedTargetRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

modelo_demo = TransformedTargetRegressor(
    regressor    = Ridge(),
    func         = np.log1p,
    inverse_func = np.expm1,
)

# Scorer RMSLE: calcula RMSE sobre log1p(y) e log1p(ŷ)
def rmsle_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    y_pred = np.maximum(y_pred, 0)  # garante ≥0 antes do log
    return -np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y))**2))

scores = cross_val_score(modelo_demo, X_proc, y, cv=5, scoring=rmsle_scorer)
print(f'RMSLE (CV-5) com Ridge + TransformedTargetRegressor: {-scores.mean():.4f} ± {scores.std():.4f}')
print('(Referência do professor com Linear Regression: 0.1754)')

RMSLE (CV-5) com Ridge + TransformedTargetRegressor: 0.1482 ± 0.0295
(Referência do professor com Linear Regression: 0.1754)


---
## 5. Prevenção de Data Leakage

### O problema

*Data leakage* ocorre quando informação do conjunto de validação "vaza" para o treinamento, gerando estimativas de desempenho otimistas. O risco concreto aqui:

- **Imputação:** se calcularmos a mediana sobre **todo** o dataset antes do split, a mediana já "conhece" os dados de validação → os folds têm uma imputação levemente melhor do que teriam em produção.
- **Escala:** se o `StandardScaler` for ajustado sobre o dataset inteiro, a média/desvio dos dados de validação influencia a normalização dos dados de treino.
- **OHE:** menos crítico, mas categorias raras podem ser contadas de forma diferente.

### A solução: tudo dentro do `Pipeline`

```
Pipeline(
    steps=[
        ('features', FunctionTransformer(engenharia_features)),  # fase 3
        ('prep',     preprocessor),                              # definido acima
        ('model',    TransformedTargetRegressor(...)),           # fase 3
    ]
)
```

Quando o scikit-learn executa `cross_val_score` ou `GridSearchCV` sobre esse pipeline:
1. Divide os índices em treino/validação.
2. Chama `.fit()` **apenas sobre o fold de treino** — mediana, média, desvio-padrão e categorias são aprendidos **só com os dados de treino**.
3. Chama `.transform()` no fold de validação usando os parâmetros aprendidos no passo 2.

Isso replica exatamente o cenário de produção: o modelo nunca vê os dados de teste antes de ser ajustado.

---
## Conclusão

O pré-processamento está completo e pronto para a **Fase 3 — Modelagem**. Resumo do que foi definido:

| Componente | Decisão |
|-----------|--------|
| Feature engineering | 10 features derivadas via `engenharia_features()` (área total, banheiros, idades, flags binárias) |
| Imputação numérica | `SimpleImputer(strategy='median')` |
| Imputação categórica | `SimpleImputer(strategy='constant', fill_value='Ausente')` |
| Encoding categórico | `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` |
| Escala numérica | `StandardScaler` |
| Transformação do alvo | `TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1)` |
| Leakage prevention | Tudo encapsulado em `Pipeline` — fit apenas no fold de treino |

Na Fase 3, o `preprocessor` definido aqui será importado diretamente e composto com o modelo final (XGBoost / LightGBM) dentro de um `Pipeline` completo.